# Rule-based control vs the MILP

A perfect-foresight MILP is not a product. A household buys an inverter that runs a *rule*:
it looks at this interval, decides a setpoint, and never sees tomorrow. This notebook asks
what that costs.

Eight rule-based controllers are executed over a full calendar year and priced through the
**same settlement the MILP is priced through** — the same battery envelope, the same agreed
billing power, the same running excess-power peak, the same `calculate_interval_price`. The
only thing that differs between a rule and the MILP is how the setpoint is chosen, so the
gap between them is a property of the controller and not of the plumbing.

Four axes, crossed:

| axis | values |
|---|---|
| controller | 8 rules, plus a no-battery floor and the whole-year MILP ceiling |
| battery | 5, 10, 20, 30 kWh nameplate |
| household | 8 Fluvius groups (PV / heat pump / EV, in every combination), 5 households each |
| price list | 4 GEN-I products, eligibility enforced |

The headline number is **`Gap_pct`**: the share of the *achievable* saving a controller gives
up. 0 % is the MILP, 100 % is a battery that might as well not be installed. Measured that
way rather than in euros, because the euros at stake differ by an order of magnitude between
a no-PV flat-rate household and a PV household on the dynamic list.

In [ ]:
### Imports
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter

import Data_Loader as dl
import Environment as environment_module
import MILP_Household as milp_module
import Horizon_Comparison as hc
import Rule_Based_Control as rbc

# Dependencies before dependents, so a reload cannot leave two modules
# disagreeing about the battery or the tariff.
dl = importlib.reload(dl)
environment_module = importlib.reload(environment_module)
milp_module = importlib.reload(milp_module)
hc = importlib.reload(hc)
rbc = importlib.reload(rbc)

print("Imports ready.")

## 1. Configuration

In [ ]:
### Study axes -------------------------------------------------------------
# Read off the modules rather than retyped: the controllers, the capacities and
# the eligibility rules are the study, and a second copy of them here would be a
# second study.
CONTROLLERS = rbc.CONTROLLER_ORDER          # no_battery, 8 rules, milp_full_period
POLICIES = rbc.POLICY_ORDER
CAPACITIES_KWH = rbc.CAPACITIES_KWH
DATASET_GROUPS = hc.DATASET_GROUPS
TARIFF_ORDER = hc.TARIFF_ORDER
PER_GROUP = hc.HOUSEHOLDS_PER_GROUP

### Battery, tariff and solver ----------------------------------------------
# All of these come from MILP_Household via Horizon_Comparison, which is where
# every study in the repository reads them from.
CHARGE_EFFICIENCY = hc.CHARGE_EFFICIENCY
DISCHARGE_EFFICIENCY = hc.DISCHARGE_EFFICIENCY
C_RATE = hc.C_RATE
INVERTER_MAX_KW = hc.INVERTER_MAX_KW
SOC_FRACTION = hc.SOC_FRACTION              # every run starts here; the MILP also ends here
MAX_DAILY_CYCLES = hc.MAX_DAILY_CYCLES
SOC_MIN_FRAC, SOC_MAX_FRAC = hc.SOC_MIN_FRAC, hc.SOC_MAX_FRAC

PRICING_REFERENCE_YEAR = hc.PRICING_REFERENCE_YEAR
PEAK_RESET_MONTHS = hc.PEAK_RESET_MONTHS
SOLVER_GAP_REL = hc.FULL_PERIOD_GAP_REL
SOLVER_TIME_LIMIT_S = hc.FULL_PERIOD_TIME_LIMIT_S

### The one household every illustration is drawn on ------------------------
TRACE_DATASET, TRACE_HOUSEHOLD = "Fluvius_PV", 1
TRACE_TARIFF = "Dinamični"       # the list with the widest spread, so the rules separate
TRACE_CAPACITY_KWH = 10.0

### Caching -----------------------------------------------------------------
RESULTS_DIR = rbc.RESULTS_DIR
RUN_BATCH = False                # True -> re-solve everything (~1.5 h on 10 workers)
N_WORKERS = 10

print(f"{'Controllers':32}{len(POLICIES)} rules + no-battery floor + MILP ceiling")
for name in POLICIES:
    policy = rbc.make_policy(name)
    flag = "" if policy.causal else "   (not deployable: reads the future)"
    print(f"{'  ' + name:32}{policy.label}{flag}")
print(f"{'Capacities':32}{', '.join(f'{c:g}' for c in CAPACITIES_KWH)} kWh nameplate")
print(f"{'Household groups':32}{len(DATASET_GROUPS)} x {PER_GROUP} households")
print(f"{'Price lists':32}{', '.join(TARIFF_ORDER)}")
print(f"{'Jobs':32}{len(rbc.study_jobs())} (household, price list) pairs after eligibility")
print(f"{'Battery':32}C-rate {C_RATE}, inverter {INVERTER_MAX_KW:g} kW, "
      f"eta {CHARGE_EFFICIENCY}/{DISCHARGE_EFFICIENCY}, start SOC {SOC_FRACTION:.0%}")
print(f"{'Envelope':32}SOC {SOC_MIN_FRAC:.0%}-{SOC_MAX_FRAC:.0%}, "
      f"daily cycles {MAX_DAILY_CYCLES}")
print(f"{'Pricing':32}{PRICING_REFERENCE_YEAR} regime, peak reset {PEAK_RESET_MONTHS} month(s)")
print(f"{'Solver':32}gap {SOLVER_GAP_REL}, limit {SOLVER_TIME_LIMIT_S}s")
print(f"{'Agreed power':32}{hc.AGREED_POWER_TAG}")

In [ ]:
### Shared chart styling
# ---------------------------------------------------------------------------
# What the colour channel means, and it means only this:
#
#   hue      what the controller READS. Three families and nothing else:
#            green  the roof      -- only ever moves the household's own PV
#            blue   the price     -- trades with the grid on a time or price signal
#            orange the meter     -- trades to hold the grid draw down
#            ink    a reference   -- the no-battery floor and the MILP ceiling
#   shade    position inside the family, light -> dark. It is NOT rank: adding
#            or dropping a controller must not repaint the survivors.
#
# Ten controllers is far too many for ten hues, and the family is the question
# every chapter actually asks. Identity is never colour-alone -- every bar is
# labelled with its controller name.
INK, INK_2, MUTED, SURFACE = "#0b0b0b", "#52514e", "#8b8a84", "#fcfcfb"
FAMILY_COLOR = {"roof": "#1baf7a", "price": "#2a78d6", "meter": "#eb6834",
                "reference": INK_2}
FAMILY_LABEL = {"roof": "reads the roof", "price": "reads the price",
                "meter": "reads the meter", "reference": "reference"}
FAMILY = {
    "no_battery": "reference",
    "self_consumption": "roof",
    "delayed_pv_charge": "roof",
    "fixed_schedule": "price",
    "price_threshold": "price",
    "price_rank_daily": "price",
    "price_oracle": "price",
    "peak_shaving": "meter",
    "self_consumption_peak_shaving": "meter",
    "milp_full_period": "reference",
}
CONTROLLER_LABEL = {"no_battery": "No battery", "milp_full_period": "MILP (whole year)"}
CONTROLLER_LABEL.update({name: rbc.make_policy(name).label for name in POLICIES})

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK_2,
    "axes.titlecolor": INK, "axes.titlesize": 12, "axes.titleweight": "semibold",
    "axes.titlelocation": "left", "axes.titlepad": 12,
    "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK_2, "ytick.color": INK_2, "font.size": 10,
    "legend.frameon": False, "figure.dpi": 110,
})

EUR = FuncFormatter(lambda v, _: f"{v:,.0f}")
PCT = FuncFormatter(lambda v, _: f"{v:,.0f}%")


def _mix(hex_color, other, t):
    """Blend two hex colours, t = share of `other`."""
    a, b = np.array(to_rgb(hex_color)), np.array(to_rgb(other))
    return tuple((1.0 - t) * a + t * b)


def controller_color(name):
    """Family hue, shaded by position WITHIN the family, never by rank."""
    family = FAMILY.get(name, "reference")
    members = [c for c in CONTROLLERS if FAMILY.get(c) == family]
    if len(members) < 2:
        return FAMILY_COLOR[family]
    t = members.index(name) / (len(members) - 1)
    return _mix(FAMILY_COLOR[family], "#ffffff", 0.45 * (1.0 - t))


def family_legend(ax, families=("roof", "price", "meter", "reference"), **kw):
    """One swatch per family: the only categorical question the charts ask."""
    ax.legend(handles=[Patch(facecolor=FAMILY_COLOR[f], label=FAMILY_LABEL[f])
                       for f in families], **kw)


def group_label(dataset):
    """Household type as a chart label: the `Fluvius` prefix carries no meaning."""
    return "basic" if dataset == "Fluvius" else str(dataset).replace("Fluvius_", "")


def finish(ax, title, xlabel=None, ylabel=None, subtitle=None):
    """House chart furniture: left-aligned title, an optional standfirst, labels."""
    # The standfirst sits between the title and the axes, so the title has to
    # be padded out of its way -- matplotlib does not reserve the space.
    ax.set_title(title, pad=26 if subtitle else 12)
    if subtitle:
        ax.text(0.0, 1.012, subtitle, transform=ax.transAxes, ha="left", va="bottom",
                fontsize=9, color=MUTED)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    return ax


def bar_labels(ax, bars, values, fmt="{:.0f}", pad=0.01, horizontal=True):
    """Direct labels on the data end, in ink rather than in the series colour."""
    span = max(abs(v) for v in values) or 1.0
    for bar, value in zip(bars, values):
        if not np.isfinite(value):
            continue
        if horizontal:
            ax.text(value + np.sign(value or 1) * pad * span, bar.get_y() + bar.get_height() / 2,
                    fmt.format(value), va="center",
                    ha="left" if value >= 0 else "right", fontsize=9, color=INK_2)
        else:
            ax.text(bar.get_x() + bar.get_width() / 2, value + np.sign(value or 1) * pad * span,
                    fmt.format(value), ha="center",
                    va="bottom" if value >= 0 else "top", fontsize=9, color=INK_2)


print("Chart styling ready.")

## 2. The controllers

Every rule sees the same three things a real inverter sees: the household load and the roof
this interval, the delivered price of a kWh, and the calendar. None of them sees the future,
with one deliberate exception.

`price_oracle` is `price_threshold` with its window pointed **forward** instead of backward —
the same quantiles, the same breakeven gate, the same everything else. It is not a
recommendation and could not be installed; it is there so that the difference between the two
isolates what foresight alone is worth to a threshold rule, with the shape of the rule held
fixed.

It is worth knowing in advance that the difference is small, and that it is not even always
positive (verification 4). A threshold rule is a heuristic: better information moves where its
thresholds sit, but it cannot make the logic behind them any less crude.

In [ ]:
roster = pd.DataFrame([
    {"Controller": name,
     "Family": FAMILY_LABEL[FAMILY[name]],
     "Causal": "yes" if rbc.make_policy(name).causal else "NO",
     "Reads": reads,
     "What it does": what}
    for name, reads, what in [
        ("self_consumption", "roof",
         "Charge the PV surplus, discharge into the deficit. Never trades with the grid."),
        ("delayed_pv_charge", "roof, yesterday's roof",
         "Throttles the morning so the pack still has headroom for the midday PV peak. "
         "Persistence forecast."),
        ("fixed_schedule", "clock",
         "Charge 01-05 local, discharge 18-22 local, soak surplus in between. No price input."),
        ("price_threshold", "price, trailing 7 d",
         "Buy under the 25th percentile of recent delivered rates, sell over the 75th, "
         "gated on round-trip breakeven."),
        ("price_rank_daily", "price, day-ahead",
         "Rank the day's 96 intervals, pair cheapest with dearest while the pair clears "
         "round-trip losses."),
        ("peak_shaving", "meter, tariff",
         "Hold the draw under min(agreed power, own 98th percentile). Stands down once the "
         "month's peak is sunk."),
        ("self_consumption_peak_shaving", "roof, meter, tariff",
         "Self-consumption over a reserve sized to the worst daily shave the trailing window "
         "demanded."),
        ("price_oracle", "price, FORWARD 7 d",
         "price_threshold with the window reversed. Diagnostic only."),
    ]
]).set_index("Controller").reindex(POLICIES)
display(roster)

### The rules on one summer day

One household, one day, one pack. The top panel is the price the rules are deciding against,
the middle one is the household they are controlling, and the bottom one is where each of them
put the charge. The MILP is the dashed trace — the only one that was allowed to see how the day
ends before deciding how it starts.

In [ ]:
TRACE_DAY = "2024-06-12"          # a clear summer day: PV, evening peak, wide SIPX spread

trace_data = hc.load_user(TRACE_HOUSEHOLD, TRACE_DATASET)
trace_env = hc.build_env(trace_data, capacity_kwh=TRACE_CAPACITY_KWH, tariff=TRACE_TARIFF)
trace_sig = rbc.build_signals(trace_env)

traces = {}
for name in POLICIES:
    out = rbc.run_policy(trace_env, rbc.make_policy(name), signals=trace_sig, keep_traces=True)
    traces[name] = {"soc": out["_soc_trace"], "setpoint": out["_setpoints"]}

milp = hc.run_strategy(trace_env, "period", "block", solver=hc.full_period_solver())
traces["milp_full_period"] = {"soc": milp["_soc_trace"], "setpoint": None}

print(f"{TRACE_DATASET} #{TRACE_HOUSEHOLD} | {TRACE_TARIFF} | {TRACE_CAPACITY_KWH:g} kWh")

In [ ]:
# Sliced on the LOCAL day. The Fluvius stamps carry a `Z` suffix but are
# Brussels-local, so a window cut on the raw index starts two hours off and the
# hour axis would wrap mid-chart.
_, days_sorted, _ = milp_module.day_calendar(trace_env.dataset.index[:trace_sig.n_steps])
day = np.asarray(trace_sig.day_steps[days_sorted.index(pd.Timestamp(TRACE_DAY).date())])
hours = trace_sig.local_hour[day]
assert np.all(np.diff(hours) > 0), "the local day did not come out monotone"

fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True,
                         gridspec_kw={"height_ratios": [1.0, 1.3, 1.3]})

# --- what the day costs ----------------------------------------------------
ax = axes[0]
ax.plot(hours, trace_sig.import_rate[day], lw=2, color=INK, label="import")
ax.plot(hours, trace_sig.export_credit[day], lw=2, color=MUTED, ls="--", label="export credit")
ax.margins(y=0.30)                       # headroom, so the legend never sits on a line
ax.legend(loc="upper left", ncols=2)
finish(ax, "The price signal every rule is deciding against",
       ylabel="EUR / kWh",
       subtitle=f"{TRACE_TARIFF}, {TRACE_DAY} — delivered, VAT included")

# --- the household ---------------------------------------------------------
ax = axes[1]
ax.fill_between(hours, 0, trace_sig.generation[day] / trace_sig.hours,
                color=_mix(FAMILY_COLOR["roof"], "#ffffff", 0.6), lw=0, label="PV")
ax.plot(hours, trace_sig.consumption[day] / trace_sig.hours, lw=2, color=INK, label="load")
ax.margins(y=0.30)
ax.set_ylim(bottom=0)                    # a negative kW here would be a drawing artefact
ax.legend(loc="upper left", ncols=2)
finish(ax, "The household the rules are controlling", ylabel="kW")

# --- where the charge sits -------------------------------------------------
ax = axes[2]
for name in POLICIES:
    ax.plot(hours, traces[name]["soc"][day], lw=2, color=controller_color(name),
            label=CONTROLLER_LABEL[name])
ax.plot(hours, traces["milp_full_period"]["soc"][day], lw=2.2, ls="--", color=INK,
        label=CONTROLLER_LABEL["milp_full_period"])
ax.set_ylim(0, TRACE_CAPACITY_KWH * 1.02)
ax.set_xticks(range(0, 25, 3))
finish(ax, "State of charge", xlabel="hour of the local day", ylabel="kWh stored",
       subtitle="dashed = perfect foresight; every solid line decided one interval at a time")

# Ten traces will not fit beside the data, so the legend goes under the figure.
fig.tight_layout(rect=(0, 0.10, 1, 1))
fig.legend(*ax.get_legend_handles_labels(), loc="lower center", ncols=4,
           fontsize=9, bbox_to_anchor=(0.5, 0.005))
plt.show()

## 3. The batch

One CSV per (household, price list), every capacity and every controller inside it. Rows carry
the ratchet reset and agreed-power rule they were priced under, and `collect_results` drops any
that were written under a superseded one — a bill computed under a different network charge is
a different quantity, not an older one.

Set `RUN_BATCH = True` in the configuration cell to recompute. It takes roughly an hour and a
half on ten workers, almost all of it in the MILP arm; the eight rules together cost about
twelve seconds per household.

In [ ]:
if RUN_BATCH:
    rbc.run_batch(n_workers=N_WORKERS)

raw = rbc.collect_results()
if raw.empty:
    raise FileNotFoundError(
        f"No results in {RESULTS_DIR}. Set RUN_BATCH = True, or run\n"
        f"    .venv/bin/python Rule_Based_Control.py --workers {N_WORKERS}"
    )

# A unit is one (household, price list, capacity) that carries BOTH references.
# A job still mid-flight has some controllers and not others, and averaging over
# a partial unit would compare a controller against a missing optimum.
complete = (
    raw.groupby(rbc.KEY_COLUMNS)["Controller"]
    .transform(lambda s: set(CONTROLLERS).issubset(set(s)))
)
if not complete.all():
    print(f"dropped {int((~complete).sum())} rows from "
          f"{raw.loc[~complete, rbc.KEY_COLUMNS].drop_duplicates().shape[0]} "
          f"incomplete units still being solved")
results = raw[complete].reset_index(drop=True)

scored = rbc.score(results)
# Redni 2T is a no-PV-only product, so it may never enter a mean taken ACROSS
# household groups: that would compare who is allowed to sign what, not which
# controller is better. Chapter 7 reports it on its own.
cross = rbc.cross_group_frame(scored)

units = results[rbc.KEY_COLUMNS].drop_duplicates()
print(f"{len(results):,} rows | {len(units):,} complete units "
      f"({units['Dataset'].nunique()} groups x {units['Household'].nunique()} households "
      f"x {units['Tariff'].nunique()} lists x {units['Capacity_kWh'].nunique()} capacities)")
print(f"peak reset {sorted(set(results['Peak_Reset']))} | "
      f"agreed power {sorted(set(results['Agreed_Power']))}")

## 4. Which controller

`Gap_pct` is the share of the achievable saving a controller gives up, where "achievable" is
what the whole-year MILP actually captures on that same household, list and pack. The mean is
taken over units; `Gap_pct_pooled` weights each unit by the euros at stake instead, which is
the honest figure when one household's battery is worth 500 EUR a year and another's is worth
five.

Redni 2T is excluded from every cross-group figure in this chapter — see chapter 7.

In [ ]:
ranking, _ = rbc.summarize(cross, by=["Controller"])
display(
    ranking[["Units", "Gap_pct", "Gap_pct_pooled", "Worst_Gap_pct", "Best_Gap_pct",
             "Savings_EUR", "Gap_EUR", "Cycles", "Grid_Charged_kWh"]]
    .rename(columns={"Savings_EUR": "Saving_EUR/a", "Grid_Charged_kWh": "GridChg_kWh/a"})
    .round(1)
)

In [ ]:
rules = [c for c in POLICIES]
gap = ranking.loc[rules, "Gap_pct"]
pooled = ranking.loc[rules, "Gap_pct_pooled"]
order = gap.sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 4.6))
y = np.arange(len(order))
bars = ax.barh(y, gap[order], height=0.62,
               color=[controller_color(c) for c in order])
# The pooled figure as a tick, so the euro-weighted answer sits beside the
# per-household one instead of in a second chart with a second scale.
ax.scatter(pooled[order], y, s=42, color=INK, zorder=3, label="pooled by euros at stake")
ax.set_yticks(y, [CONTROLLER_LABEL[c] for c in order])
ax.xaxis.set_major_formatter(PCT)
# A label COLUMN rather than a label on each data end: the pooled dots sit close
# to the bar ends and the two would collide on the rules that rank best.
label_x = float(gap.max()) * 1.04
ax.set_xlim(0, label_x * 1.10)
for row, value in zip(y, gap[order]):
    ax.text(label_x, row, f"{value:.0f}%", va="center", ha="left",
            fontsize=9, color=INK_2)
# Outside the axes: inside, the legend's sample dot reads as a data point.
ax.legend(loc="lower right", bbox_to_anchor=(1.0, 1.0))
finish(ax, "How much of the achievable saving each rule gives up",
       xlabel="gap to the whole-year MILP  [% of achievable saving]",
       subtitle="0 % = perfect foresight, 100 % = the battery earns nothing. "
                "Lower is better.")
fig.tight_layout()
plt.show()

### The same question per price list

A controller is not good or bad in the abstract: it is good or bad against a price signal. A
rule that never trades with the grid cannot lose on a flat list and cannot win on a volatile
one.

In [ ]:
by_tariff, _ = rbc.summarize(scored, by=["Tariff", "Controller"])
table = (by_tariff["Gap_pct"].unstack("Tariff")
         .reindex(index=POLICIES, columns=TARIFF_ORDER))

fig, ax = plt.subplots(figsize=(10, 5))
width = 0.8 / len(TARIFF_ORDER)
x = np.arange(len(POLICIES))
for k, tariff in enumerate(TARIFF_ORDER):
    values = table[tariff].to_numpy(dtype=float)
    offset = (k - (len(TARIFF_ORDER) - 1) / 2) * width
    ax.bar(x + offset, values, width=width * 0.92,
           color=_mix(INK_2, "#ffffff", 0.72 - 0.22 * k), label=tariff)
ax.set_xticks(x, [CONTROLLER_LABEL[c] for c in POLICIES], rotation=30, ha="right")
ax.yaxis.set_major_formatter(PCT)
ax.legend(loc="upper left", ncols=len(TARIFF_ORDER), title="price list")
finish(ax, "The gap depends on what the price signal offers",
       ylabel="gap to the MILP  [%]",
       subtitle="Redni 2T is no-PV only, so its bars cover a different set of households")
fig.tight_layout()
plt.show()
display(table.round(1))

### The best and the worst household

A mean hides the two cases worth looking at. Both are drawn only from units where there was a
real prize to compete for: on a household whose whole achievable saving is four euros, a gap of
several hundred percent is rounding, not a controller failing, and those units would otherwise
occupy the entire "worst" half of the table.

In [ ]:
# A percentage is only worth reading where there was a prize to win. Units whose
# whole achievable saving is a few euros produce enormous gaps out of rounding,
# and they would otherwise fill the "worst" half of this table every time.
MEANINGFUL_PRIZE_EUR = 25.0

best_rule = ranking.loc[rules, "Gap_pct"].idxmin()
focus = (cross[(cross["Controller"] == best_rule)
               & (cross["Achievable_EUR"] >= MEANINGFUL_PRIZE_EUR)]
         .dropna(subset=["Gap_pct"]))
extremes = pd.concat([focus.nsmallest(3, "Gap_pct"), focus.nlargest(3, "Gap_pct")])
print(f"{len(focus)} of {int((cross['Controller'] == best_rule).sum())} units had at "
      f"least {MEANINGFUL_PRIZE_EUR:.0f} EUR/a of achievable saving to compete for")

print(f"Best rule overall: {CONTROLLER_LABEL[best_rule]} "
      f"({ranking.loc[best_rule, 'Gap_pct']:.0f}% mean gap)\n")
display(
    extremes[["Dataset", "Household", "Tariff", "Capacity_kWh", "Gap_pct",
              "Savings_EUR", "Achievable_EUR", "No_Battery_EUR", "Grid_Charged_kWh"]]
    .assign(Dataset=lambda d: d["Dataset"].map(group_label))
    .round(1).set_index(["Dataset", "Household", "Tariff", "Capacity_kWh"])
)

lo, hi = extremes.iloc[0], extremes.iloc[-1]
print(f"best  {group_label(lo.Dataset)} #{int(lo.Household)} on {lo.Tariff} at "
      f"{lo.Capacity_kWh:g} kWh: captures {100 - lo.Gap_pct:.0f}% of a "
      f"{lo.Achievable_EUR:,.0f} EUR/a prize")
print(f"worst {group_label(hi.Dataset)} #{int(hi.Household)} on {hi.Tariff} at "
      f"{hi.Capacity_kWh:g} kWh: captures {100 - hi.Gap_pct:.0f}% of a "
      f"{hi.Achievable_EUR:,.0f} EUR/a prize")

## 5. Battery size

Does a rule fall further behind as the pack grows? It should. A bigger battery is a longer
lever on the future, and a controller that decides one interval at a time has no more future
to pull on than it had at 5 kWh.

In [ ]:
by_capacity, _ = rbc.summarize(cross, by=["Capacity_kWh", "Controller"])
gap_by_cap = by_capacity["Gap_pct"].unstack("Controller").reindex(columns=POLICIES)
save_by_cap = (rbc.summarize(cross, by=["Capacity_kWh", "Controller"])[0]["Savings_EUR"]
               .unstack("Controller").reindex(columns=POLICIES))
milp_saving = (cross[cross["Controller"] == "milp_full_period"]
               .groupby("Capacity_kWh")["Savings_EUR"].mean())

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

ax = axes[0]
for name in POLICIES:
    ax.plot(gap_by_cap.index, gap_by_cap[name], marker="o", ms=5, lw=2,
            color=controller_color(name), label=CONTROLLER_LABEL[name])
ax.yaxis.set_major_formatter(PCT)
ax.set_xticks(CAPACITIES_KWH)
finish(ax, "The gap widens with the pack", xlabel="nameplate capacity [kWh]",
       ylabel="gap to the MILP  [%]",
       subtitle="a bigger battery is a longer lever on a future the rule cannot see")

ax = axes[1]
for name in POLICIES:
    ax.plot(save_by_cap.index, save_by_cap[name], marker="o", ms=5, lw=2,
            color=controller_color(name))
ax.plot(milp_saving.index, milp_saving, lw=2.2, ls="--", color=INK, marker="D", ms=5,
        label=CONTROLLER_LABEL["milp_full_period"])
ax.axhline(0, color=MUTED, lw=1)
ax.yaxis.set_major_formatter(EUR)
ax.set_xticks(CAPACITIES_KWH)
ax.legend(loc="upper left")
finish(ax, "What each rule actually earns", xlabel="nameplate capacity [kWh]",
       ylabel="saving vs no battery [EUR/a]",
       subtitle="mean over households and price lists")

# Four of the eight lines are blue, so the family legend alone cannot identify a
# line. The controllers get a real legend under both panels instead.
handles, labels_ = axes[1].get_legend_handles_labels()
axes[1].get_legend().remove()
fig.tight_layout(rect=(0, 0.16, 1, 1))
fig.legend(
    handles=[Line2D([], [], color=controller_color(c), lw=2, label=CONTROLLER_LABEL[c])
             for c in POLICIES] + handles,
    loc="lower center", ncols=3, fontsize=9, bbox_to_anchor=(0.5, 0.0),
)
plt.show()
display(gap_by_cap.round(1))

## 6. Household type

Eight Fluvius groups: a plain load, and every combination of a PV roof, a heat pump and an
electric vehicle. What a controller can do depends on what the household gives it to work
with — a roof to store, or a big flexible load to move.

In [ ]:
by_group, _ = rbc.summarize(cross, by=["Dataset", "Controller"])
heat = (by_group["Gap_pct"].unstack("Controller")
        .reindex(index=[g for g in DATASET_GROUPS], columns=POLICIES))

fig, ax = plt.subplots(figsize=(11, 5.2))
values = heat.to_numpy(dtype=float)
# One hue, light -> dark: this panel encodes magnitude, not identity.
image = ax.imshow(values, aspect="auto", cmap="Blues", vmin=0,
                  vmax=float(np.nanmax(values)))
ax.set_xticks(range(len(POLICIES)), [CONTROLLER_LABEL[c] for c in POLICIES],
              rotation=30, ha="right")
ax.set_yticks(range(len(heat.index)), [group_label(g) for g in heat.index])
ax.grid(False)
mid = float(np.nanmax(values)) * 0.55
for i in range(values.shape[0]):
    for j in range(values.shape[1]):
        v = values[i, j]
        if not np.isfinite(v):
            continue
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=9,
                color=SURFACE if v > mid else INK)
fig.colorbar(image, ax=ax, label="gap to the MILP [%]", pad=0.02)
finish(ax, "Gap to the MILP by household type",
       subtitle="darker is a bigger share of the achievable saving given up. "
                "Redni 2T excluded, so every row covers the same three price lists.")
fig.tight_layout()
plt.show()

In [ ]:
# What each group had to gain in the first place, so a large gap can be read
# against the size of the prize rather than in isolation.
prize = (cross[cross["Controller"] == "milp_full_period"]
         .groupby("Dataset")
         .agg(Achievable_EUR=("Achievable_EUR", "mean"),
              No_Battery_EUR=("No_Battery_EUR", "mean"))
         .reindex(DATASET_GROUPS))

fig, ax = plt.subplots(figsize=(10, 4.2))
x = np.arange(len(prize))
bars = ax.bar(x, prize["Achievable_EUR"], width=0.6,
              color=_mix(FAMILY_COLOR["reference"], "#ffffff", 0.45))
ax.set_xticks(x, [group_label(g) for g in prize.index], rotation=20, ha="right")
ax.yaxis.set_major_formatter(EUR)
bar_labels(ax, bars, list(prize["Achievable_EUR"]), fmt="{:,.0f}", horizontal=False)
finish(ax, "What perfect foresight is worth to each household type",
       ylabel="achievable saving [EUR/a]",
       subtitle="mean over price lists and capacities — the denominator every "
                "percentage in this notebook is taken of")
fig.tight_layout()
plt.show()
display(prize.round(0))

## 7. Pricing contract

Two things to keep apart here.

**Eligibility is not a tariff result.** Redni 2T is the plain `GENI_REDNI` list; GEN-I
publishes no two-tariff samooskrba product, so a household with a roof cannot sign it. It is
run only against the four no-PV groups, and any figure that averages it together with the
three samooskrba lists is comparing who may sign what.

**A saving is not necessarily a battery.** A list with a wide fixed block spread — Aktivni
buys at 0.04090 and sells at 0.14990 — makes grid-to-battery arbitrage pay on its own, every
day, with no PV involved at all. That is a real saving, but it is a trading result, not a
self-consumption one, and the `Grid_Charged_kWh` share is what tells the two apart.

In [ ]:
share = (
    scored.assign(
        Grid_Share=lambda d: np.where(d["Charged_kWh"] > 1e-9,
                                      100.0 * d["Grid_Charged_kWh"] / d["Charged_kWh"],
                                      np.nan))
    .groupby(["Tariff", "Controller"])
    .agg(Saving_EUR=("Savings_EUR", "mean"),
         Grid_Share_pct=("Grid_Share", "mean"),
         Cycles=("Equivalent_Full_Cycles", "mean"))
)
grid_share = share["Grid_Share_pct"].unstack("Tariff").reindex(
    index=POLICIES + ["milp_full_period"], columns=TARIFF_ORDER)
saving = share["Saving_EUR"].unstack("Tariff").reindex(
    index=POLICIES + ["milp_full_period"], columns=TARIFF_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
y = np.arange(len(grid_share.index))
labels = [CONTROLLER_LABEL[c] for c in grid_share.index]

for ax, frame, title, formatter, unit, note in (
    (axes[0], saving, "What the battery earns", EUR, "EUR/a",
     "a tall bar here beside a tall bar there is arbitrage, not self-consumption"),
    (axes[1], grid_share, "How much of it was bought from the grid", PCT, "%",
     "100 % means every stored kWh was bought, not harvested"),
):
    height = 0.8 / len(TARIFF_ORDER)
    for k, tariff in enumerate(TARIFF_ORDER):
        offset = (k - (len(TARIFF_ORDER) - 1) / 2) * height
        # The price list is the SECOND categorical question these panels ask, and
        # the hue channel is already spoken for by the controller family, so it
        # gets a neutral ramp plus a legend rather than four more hues.
        ax.barh(y + offset, frame[tariff].to_numpy(dtype=float), height=height * 0.9,
                color=_mix(INK_2, "#ffffff", 0.72 - 0.22 * k), label=tariff)
    ax.axvline(0, color=MUTED, lw=1)
    ax.xaxis.set_major_formatter(formatter)
    finish(ax, title, xlabel=unit, subtitle=note)
axes[0].set_yticks(y, labels)
# Under the figure: the right panel's bars run the full width, so a legend
# placed inside it would sit on top of four of them.
handles, legend_labels = axes[1].get_legend_handles_labels()
fig.tight_layout(rect=(0, 0.09, 1, 1))
fig.legend(handles, legend_labels, loc="lower center", ncols=len(TARIFF_ORDER),
           title="price list", bbox_to_anchor=(0.5, 0.0))
plt.show()
display(saving.round(0))
print("A controller that never grid-charges shows a zero-length bar on the right: "
      "self-consumption and delayed PV charge store only what the roof gave them.")

### Do two lists produce the same dispatch?

Only price *spreads* drive a battery. Two lists that sit a flat amount apart hand the
controller the same decision at every interval and therefore the same trajectory — the bills
differ, the dispatch does not. Worth checking before reading anything into two lists that
track each other.

In [ ]:
dispatch = (
    scored[scored["Controller"].isin(POLICIES)]
    .pivot_table(index=["Dataset", "Household", "Capacity_kWh", "Controller"],
                 columns="Tariff", values="Discharged_kWh")
    .reindex(columns=TARIFF_ORDER)
)
pairs = []
for i, a in enumerate(TARIFF_ORDER):
    for b in TARIFF_ORDER[i + 1:]:
        both = dispatch[[a, b]].dropna()
        if both.empty:
            continue
        scale = both.to_numpy().max() or 1.0
        pairs.append({"A": a, "B": b, "Units": len(both),
                      "Max_diff_kWh": float((both[a] - both[b]).abs().max()),
                      "Rel_diff_pct": float(100.0 * (both[a] - both[b]).abs().max() / scale)})
display(pd.DataFrame(pairs).round(2).set_index(["A", "B"]))
print("Rel_diff_pct near zero means the two lists produce ONE trajectory and only "
      "the bill differs.")

## 8. Peak and the excess-power charge

The energy bill is not the only thing a battery can move. `omrežnina za presežno moč` is
charged on the running monthly peak per network block above the agreed billing power, and it
is the one part of the bill a rule can attack without knowing anything about prices — a kW
meter is enough.

It is also the line where a simple rule comes closest to the optimizer. Everywhere else the
MILP's foresight is decisive; here a controller with nothing but a kW meter lands within a few
percent of it.

The two panels disagree on purpose, and the reason is worth being precise about. The MILP
minimises the *whole* bill, so it will buy a bigger peak whenever the energy it arbitrages is
worth more than the excess charge that peak triggers — and because the charge is levied per
network block, it can push its imports into the cheap blocks and end up with the **highest
metered peak of any controller and the lowest charge for it**. A single-purpose peak shaver
makes no such trade: it holds the peak down and takes no interest in what that costs it in
energy.

In [ ]:
peak = (
    scored.groupby("Controller")
    .agg(Peak_kW=("Peak_Import_kW", "mean"),
         Power_EUR=("Power_EUR", "mean"),
         Energy_EUR=("Energy_EUR", "mean"),
         Cycles=("Equivalent_Full_Cycles", "mean"))
    .reindex(CONTROLLERS)
)
floor_kw = peak.loc["no_battery", "Peak_kW"]
floor_eur = peak.loc["no_battery", "Power_EUR"]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
order = [c for c in CONTROLLERS if c != "no_battery"]
y = np.arange(len(order))

ax = axes[0]
bars = ax.barh(y, peak.loc[order, "Peak_kW"], height=0.62,
               color=[controller_color(c) for c in order])
ax.axvline(floor_kw, color=INK, lw=1.6, ls="--")
ax.text(floor_kw, len(order) - 0.3, "  no battery", color=INK, fontsize=9, va="center")
ax.set_yticks(y, [CONTROLLER_LABEL[c] for c in order])
bar_labels(ax, bars, list(peak.loc[order, "Peak_kW"]), fmt="{:.2f}")
finish(ax, "Peak grid import", xlabel="kW, mean over units",
       subtitle="what the ratchet is measured on")

ax = axes[1]
bars = ax.barh(y, peak.loc[order, "Power_EUR"], height=0.62,
               color=[controller_color(c) for c in order])
ax.axvline(floor_eur, color=INK, lw=1.6, ls="--")
ax.set_yticks(y, [])
bar_labels(ax, bars, list(peak.loc[order, "Power_EUR"]), fmt="{:.2f}")
finish(ax, "Excess-power charge", xlabel="EUR/a, mean over units",
       subtitle="omrežnina za presežno moč, the part a kW meter alone can attack")

fig.tight_layout()
plt.show()
display(peak.round(2))

# Stated as a computed fact rather than as prose: which controller actually pays
# the least excess-power charge, and what it gave up elsewhere to get there.
best_rule_on_power = peak.loc[POLICIES, "Power_EUR"].idxmin()
milp_power = peak.loc["milp_full_period", "Power_EUR"]
rule_power = peak.loc[best_rule_on_power, "Power_EUR"]
print(f"Excess-power charge: {peak.loc['no_battery', 'Power_EUR']:.2f} EUR/a with no battery, "
      f"{milp_power:.2f} for the MILP, {rule_power:.2f} for the best rule "
      f"({CONTROLLER_LABEL[best_rule_on_power]}).")
print(f"The rule lands within {abs(rule_power - milp_power) / max(milp_power, 1e-9):.0%} of "
      f"perfect foresight on this line, while giving up "
      f"{ranking.loc[best_rule_on_power, 'Gap_pct']:.0f}% of the saving overall — "
      f"the ratchet is the one part of the bill a kW meter is nearly enough for.")
print(f"It pays {peak.loc[best_rule_on_power, 'Energy_EUR'] - peak.loc['milp_full_period', 'Energy_EUR']:+,.0f} "
      f"EUR/a more than the MILP on energy to get there.")
print(f"Meanwhile the MILP runs the {'HIGHEST' if peak.loc['milp_full_period', 'Peak_kW'] == peak['Peak_kW'].max() else 'a lower'} "
      f"metered peak of any controller ({peak.loc['milp_full_period', 'Peak_kW']:.2f} kW against "
      f"{peak.loc['no_battery', 'Peak_kW']:.2f} kW with no battery) and still pays the least for "
      f"it — it moves its imports into the cheap network blocks. Minimising one bill line is "
      f"not minimising the bill.")

## 9. Verification

Four claims this notebook rests on, checked against the results themselves rather than
asserted in prose. The module-level invariants — the envelope, the re-pricing, the daily cycle
cap — live in `Rule_Based_Control_Check.py`:

```
.venv/bin/python Rule_Based_Control_Check.py
```

In [ ]:
### Verification 1 — the MILP is a lower bound on every rule
# The MILP minimises the same bill over the same battery with perfect foresight,
# so no rule may come in under it. A rule that does is not controlling better,
# it is being priced differently, and every number above would be void.
tolerance = scored["Optimum_EUR"].abs() * SOLVER_GAP_REL + 1e-6
under = scored[scored["Gap_to_Optimum_EUR"] < -tolerance]
assert under.empty, (
    f"{len(under)} rows beat the perfect-foresight optimum by more than the "
    f"{SOLVER_GAP_REL:.0e} MIP gap:\n{under.head()}"
)
print(f"Verification 1 OK — all {len(scored):,} rows at or above the optimum "
      f"(worst undershoot {scored['Gap_to_Optimum_EUR'].min():+.4f} EUR, "
      f"within the {SOLVER_GAP_REL:.0e} MIP gap)")

In [ ]:
### Verification 2 — every rule respected the battery envelope
# The runner clamps every setpoint to the feasible bounds before executing it,
# so a rule cannot drift the state of charge at all. The MILP arm is held to a
# looser bar on purpose: it stops at its MIP gap, and a solution that is optimal
# to 0.1 % can land a few micro-kWh outside the SOC bound.
rules_drift = scored[scored["Controller"].isin(POLICIES)]["SOC_Drift_kWh"].abs().max()
milp_drift = scored[scored["Controller"] == "milp_full_period"]["SOC_Drift_kWh"].abs().max()
assert rules_drift < 1e-9, f"a rule drifted the state of charge by {rules_drift:.2e} kWh"
assert milp_drift < 1e-3, f"the MILP arm drifted {milp_drift:.2e} kWh, beyond solver noise"

# No rule curtails, so the whole PV yield is either used, stored or exported.
curtailed = scored[scored["Controller"].isin(POLICIES)]["Curtailed_kWh"].abs().max()
assert curtailed < 1e-9, f"a rule curtailed {curtailed:.4f} kWh"
print(f"Verification 2 OK — rules drifted {rules_drift:.1e} kWh (clamped by the runner), "
      f"MILP {milp_drift:.1e} kWh (solver noise), no rule curtailed ({curtailed:.1e} kWh)")

In [ ]:
### Verification 3 — the references are consistent across a unit
# `No_Battery_EUR` is a property of the household, the list and the year, so it
# must not vary with the capacity or the controller within one household-list.
spread = (
    scored.groupby(["Dataset", "Household", "Tariff"])["No_Battery_EUR"]
    .agg(lambda s: s.max() - s.min()).max()
)
assert spread < 1e-6, f"the no-battery baseline varies by {spread:.6f} EUR within a unit"

# And self-consumption may never have touched the grid, on any household.
grid = scored[scored["Controller"] == "self_consumption"]["Grid_Charged_kWh"].abs().max()
assert grid < 1e-9, f"self_consumption grid-charged {grid:.4f} kWh"
print(f"Verification 3 OK — baseline constant within each unit ({spread:.1e} EUR), "
      f"self-consumption never grid-charged ({grid:.1e} kWh)")

In [ ]:
### Verification 4 — foresight, and nothing else, separates the two threshold rules
# `price_oracle` is `price_threshold` with the window pointed forward. Same rule,
# same quantiles, same breakeven gate. If the forward version were not at least
# as good, the pair would not be isolating foresight.
pair = (scored[scored["Controller"].isin(["price_threshold", "price_oracle"])]
        .pivot_table(index=rbc.KEY_COLUMNS, columns="Controller",
                     values="Cost_EUR_Closed").dropna())
worse = pair[pair["price_oracle"] > pair["price_threshold"] + 1e-6]
gain = pair["price_threshold"] - pair["price_oracle"]
# Foresight helps ON AVERAGE, and that is all that can be asserted. A threshold
# rule is a heuristic, not an optimizer: feeding it better information moves its
# thresholds, and a moved threshold is not guaranteed to be a better one. That
# the forward window loses on a minority of units is a property of the RULE, not
# a defect in the experiment -- and it is itself the finding, because it says the
# gap to the MILP is mostly the shape of the rule rather than the horizon.
assert gain.mean() > 0, (
    f"the forward window is worth {gain.mean():+.4f} EUR/a on average — the pair "
    f"is not isolating foresight"
)
print(f"Verification 4 OK — foresight is worth {gain.mean():+.2f} EUR/a on average "
      f"({gain.min():+.2f} .. {gain.max():+.2f} across units)")
print(f"  but it LOST on {len(worse)} of {len(pair)} units ({len(worse)/len(pair):.0%}): "
      f"better information moves a heuristic's thresholds, it does not sharpen its logic.")

## 10. Read-out

In [ ]:
rules_only = ranking.loc[POLICIES]
best = rules_only["Gap_pct"].idxmin()
worst = rules_only["Gap_pct"].idxmax()
causal_rules = [c for c in POLICIES if rbc.make_policy(c).causal]
best_causal = rules_only.loc[causal_rules, "Gap_pct"].idxmin()
foresight = (rules_only.loc["price_threshold", "Savings_EUR"]
             - rules_only.loc["price_oracle", "Savings_EUR"])
milp_row = rbc.summarize(cross, by=["Controller"])[0].loc["milp_full_period"]
peak_best = peak.loc[POLICIES, "Peak_kW"].idxmin()
cap_first, cap_last = CAPACITIES_KWH[0], CAPACITIES_KWH[-1]

lines = [
    f"{len(units):,} complete units: {units['Dataset'].nunique()} Fluvius groups x "
    f"{PER_GROUP} households x {len(TARIFF_ORDER)} price lists x "
    f"{len(CAPACITIES_KWH)} capacities, one calendar year each.",
    "",
    f"The whole-year MILP saves {milp_row['Savings_EUR']:,.0f} EUR/a on average. That is the "
    f"prize; every percentage below is a share of it.",
    "",
    f"Best deployable rule: {CONTROLLER_LABEL[best_causal]}, giving up "
    f"{rules_only.loc[best_causal, 'Gap_pct']:.0f}% of the achievable saving "
    f"({rules_only.loc[best_causal, 'Savings_EUR']:,.0f} EUR/a of "
    f"{milp_row['Savings_EUR']:,.0f}). Weighted by the euros at stake rather than by "
    f"household it gives up {rules_only.loc[best_causal, 'Gap_pct_pooled']:.0f}%.",
    f"Worst rule: {CONTROLLER_LABEL[worst]}, at "
    f"{rules_only.loc[worst, 'Gap_pct']:.0f}%.",
    f"Spread across the eight rules: {rules_only['Gap_pct'].min():.0f}% to "
    f"{rules_only['Gap_pct'].max():.0f}% — the choice of rule is worth "
    f"{rules_only['Savings_EUR'].max() - rules_only['Savings_EUR'].min():,.0f} EUR/a.",
    "",
    f"Foresight, isolated: pointing the same threshold rule's window forward instead of "
    f"backward is worth {-foresight:,.0f} EUR/a. Everything else in the gap to the MILP is "
    f"the shape of the rule, not the horizon it sees.",
    "",
    f"Pack size: the mean gap moves from "
    f"{gap_by_cap.loc[cap_first].mean():.0f}% at {cap_first:g} kWh to "
    f"{gap_by_cap.loc[cap_last].mean():.0f}% at {cap_last:g} kWh. A bigger battery is a "
    f"longer lever on a future a rule cannot see, so the rules fall further behind as it "
    f"grows.",
    "",
    f"Peak power: {CONTROLLER_LABEL[peak_best]} holds the mean peak to "
    f"{peak.loc[peak_best, 'Peak_kW']:.2f} kW against "
    f"{peak.loc['no_battery', 'Peak_kW']:.2f} kW with no battery, and the excess-power "
    f"charge from {peak.loc['no_battery', 'Power_EUR']:.2f} to "
    f"{peak.loc[peak_best, 'Power_EUR']:.2f} EUR/a. That part of the bill needs a kW meter, "
    f"not a price forecast.",
]
print("\n".join(lines))

### Caveats

- **Terminal state of charge.** Every MILP strategy is pinned to start *and* end the year at
  50 % of capacity; a rule has no terminal constraint. `Cost_EUR_Closed` — the figure every
  comparison above uses — values the shortfall at the mean delivered import rate, so a rule
  that runs the pack down in December cannot book the difference as a saving. `Cost_EUR` is
  kept in the results so the size of that adjustment stays visible.

- **Curtailment is asymmetric.** The MILP may spill PV for free; no rule here does, matching
  the no-battery baseline, which also curtails nothing. `Curtailed_kWh` is reported for both
  (verification 2). Where the MILP spills, part of its advantage is an option the rules were
  not given.

- **`price_oracle` is not a controller.** It reads seven days of future prices. It exists only
  to separate "the rule is too simple" from "the rule cannot see ahead", and it should never
  appear in a recommendation.

- **`delayed_pv_charge` is solving a problem this market does not have.** Feed-in damping was
  designed for a hard export cap (the German 70 % rule), where surplus not stored is surplus
  destroyed. Under `si_samooskrba` every exported kWh is credited, so holding back the morning
  only means exporting the cheaper midday instead. The result is a fair reading of the rule
  under this settlement, not of the rule in general.

- **Redni 2T covers different households.** It is a no-PV-only product, excluded from every
  cross-group figure (chapters 4–6, 8) and reported on its own in chapter 7.

- **One year, borrowed calendar.** Belgian Fluvius profiles for 2024 priced under the 2026
  Slovenian tariff regime. The stamps carry a `Z` suffix but are Brussels-local, which is why
  every clock rule and every daily budget here reads local time through `si_cas.v_lokalni_cas`
  rather than off the index.

- **Agreed billing power is exogenous.** Derived from the *no-battery* profile with a one-month
  lag, so a controller cannot lower next month's contracted power by shaving this month's peak.
  That is deliberate — it keeps the ratchet comparable across controllers — but it understates
  what a peak-shaving rule would be worth over several years.

- **The rules are untuned.** Every threshold, window and reserve is a stated default, not a
  fitted parameter. Tuning them per household would narrow the gap to the MILP and would also
  stop being a rule-based controller.